In [ ]:
# Bound this kernel to currently available RAM. The combiner processes one
# horizon at a time and never retains horizon-specific feature matrices.
import os
import sys

sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/"))
from memory_compute import *

install_memory_guard()


In [ ]:
import gc
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq
from IPython.display import display
from tqdm.auto import tqdm

from post_training_tail import fit_horizon_postprocessor

sys.path.append(os.path.abspath("../../"))
from EPF import variables

sys.path.append(os.path.abspath("../2_Features_build"))
import target_features


# Residual stacker with calibrated spike routing

This notebook replaces the hand-written spike/dip grid policies and final
price isotonic regression with two complementary post-training outputs per horizon:

1. a central residual stacker optimized for point-forecast MAE; and
2. a calibrated spike alert plus tail-aware price forecast optimized for F2 and
   spike magnitude while constraining ordinary-period degradation.

The eight already-trained component models are unchanged. For each horizon the
stacker receives their price predictions, classifier probabilities, logits,
expert disagreements, and probability-weighted spike/dip gaps. It learns a
correction to the clipped-L1 base forecast. This preserves the base model's
fine-grained prediction while allowing nonlinear corrections when the event
models agree that a spike or negative-price interval is likely.

Validation is divided chronologically into train, tune, and untouched audit
blocks, with horizon-sized purges at both boundaries. Complexity, alert threshold,
and tail-routing strength are chosen before the audit block. Final stackers and
probability calibrators are then refit on targets known before `TEST_START`.
No test targets are used here.


In [ ]:
SELECTED_FEATURES_DIR = variables.CWD / "4_Features_select" / "Selected_features"
STACKER_DIAGNOSTICS_PATH = (
    variables.CWD / "5_Model" / "Data" / "4_combine_models"
    / "residual_stacker_validation_diagnostics.csv"
)

FEATURE_OBJECTIVES = ["normal", "arcsinh", "spikes", "dips"]
HORIZON_LIST = list(range(1, variables.HORIZON_COUNT + 1))

# key, selected-feature objective, trained-model filename suffix, output kind
COMPONENT_SPECS = [
    ("base_l1", "normal", "full_range_regressor_clipped_MAE_loss", "regression"),
    ("base_l2", "arcsinh", "full_range_regressor_unclipped_RMSE_loss", "regression"),
    ("spike_probability", "spikes", "positive_spike_classifier_binary_loss", "probability"),
    ("spike_mae", "spikes", "positive_spike_regressor_unclipped_mae_loss", "regression"),
    ("spike_q90", "spikes", "positive_spike_regressor_unclipped_quantile_loss", "regression"),
    ("dip_probability", "dips", "negative_spike_classifer_unclipped_binary_loss", "probability"),
    ("dip_mae", "dips", "negative_spike_regressor_unclipped_mae_loss", "regression"),
    ("dip_q10", "dips", "negative_spike_regressor_unclipped_quantile_loss", "regression"),
]

META_FEATURE_NAMES = [
    "base_l1",
    "base_l2",
    "spike_mae",
    "spike_q90",
    "dip_mae",
    "dip_q10",
    "spike_probability",
    "dip_probability",
    "spike_logit",
    "dip_logit",
    "base_l2_minus_l1",
    "spike_mae_positive_gap",
    "spike_q90_positive_gap",
    "dip_mae_negative_gap",
    "dip_q10_negative_gap",
    "weighted_spike_mae_gap",
    "weighted_spike_q90_gap",
    "weighted_dip_mae_gap",
    "weighted_dip_q10_gap",
]

THREADS_PER_MODEL = max(1, (psutil.cpu_count(logical=False) or 2) - 2)
STACKER_PARAMS = {
    "objective": "regression_l1",
    "metric": "mae",
    "n_estimators": 800,
    "learning_rate": 0.03,
    "num_leaves": 15,
    "max_depth": 4,
    "min_child_samples": 500,
    "subsample": 0.80,
    "subsample_freq": 1,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.10,
    "reg_lambda": 5.0,
    "random_state": 1729,
    "force_col_wise": True,
    "verbosity": -1,
    "n_jobs": THREADS_PER_MODEL,
    "num_threads": THREADS_PER_MODEL,
}

POSTPROCESSOR_PARAMS = {
    "early_stopping_rounds": 60,
    "train_fraction": 0.60,
    "tune_fraction": 0.20,
    "spike_beta": 2.0,
    "minimum_spike_precision": 0.60,
    "spike_error_weight": 4.0,
    "max_overall_mae_degradation": 0.05,
    "max_non_spike_mae_degradation": 0.10,
}


In [ ]:
def parquet_data_columns(path):
    schema = pq.ParquetFile(path).schema_arrow
    metadata = schema.pandas_metadata or {}
    index_columns = {
        column for column in metadata.get("index_columns", [])
        if isinstance(column, str)
    }
    return [column for column in schema.names if column not in index_columns]


def read_parquet_date_slice(path, columns, start, end):
    """Read selected columns for `(start, end]` without loading the full parquet."""
    parquet = pq.ParquetFile(path, pre_buffer=False, memory_map=False)
    metadata = parquet.schema_arrow.pandas_metadata or {}
    index_columns = [
        column for column in metadata.get("index_columns", [])
        if isinstance(column, str)
    ]
    if len(index_columns) != 1:
        raise ValueError(f"Expected one stored index column in {path}; found {index_columns}")
    index_column = index_columns[0]

    available = set(parquet_data_columns(path))
    missing = sorted(set(columns).difference(available))
    if missing:
        raise KeyError(f"{path} is missing {len(missing)} requested columns; examples={missing[:5]}")

    parts = []
    for record_batch in parquet.iter_batches(
        batch_size=10_000,
        columns=list(columns) + [index_column],
        use_threads=False,
    ):
        batch = record_batch.to_pandas()
        dates = batch.index if batch.index.name == index_column else batch[index_column]
        mask = (dates > start) & (dates <= end)
        if mask.any():
            batch = batch.loc[mask]
            if index_column in batch.columns:
                batch = batch.set_index(index_column)
            parts.append(batch.astype(np.float32))

    if not parts:
        return pd.DataFrame(columns=columns, index=pd.DatetimeIndex([], name=index_column))
    result = pd.concat(parts).sort_index()
    if result.index.has_duplicates:
        raise ValueError(f"Duplicate timestamps found in {path}")
    return result


features_optimal_amount_by_objective = {
    objective: pd.read_parquet(
        SELECTED_FEATURES_DIR / f"FEATURES_OPTIMAL_AMOUNT_{objective}.parquet"
    )
    for objective in FEATURE_OBJECTIVES
}


def selected_features(objective, horizon):
    table = features_optimal_amount_by_objective[objective]
    column = f"h{horizon}"
    if column not in table:
        raise KeyError(f"Missing selection column {column} for {objective}")
    result = table.loc[table[column].astype(bool), "feature"].tolist()
    if not result:
        raise ValueError(f"No selected features for {objective} {column}")
    return result


needed_features = sorted({
    feature
    for objective in FEATURE_OBJECTIVES
    for horizon in HORIZON_LIST
    for feature in selected_features(objective, horizon)
})

target_columns = [f"target_h{h}" for h in HORIZON_LIST]
features_validate = read_parquet_date_slice(
    variables.FEATURES_DATASET_PATH,
    needed_features,
    variables.VALID_START,
    variables.TEST_START,
)
targets_validate = read_parquet_date_slice(
    variables.AGG_TARGET_DATASET_PATH,
    target_columns,
    variables.VALID_START,
    variables.TEST_START,
)

if not features_validate.index.equals(targets_validate.index):
    features_validate = features_validate.reindex(targets_validate.index)
if features_validate.isna().any().any():
    bad = features_validate.columns[features_validate.isna().any()].tolist()
    raise ValueError(f"Validation feature slice contains NaNs; examples={bad[:5]}")

print(
    f"Validation data: {len(targets_validate):,} rows, "
    f"{len(needed_features):,} distinct selected features, "
    f"{len(HORIZON_LIST)} horizons"
)


In [ ]:
def trained_model_path(horizon, model_name):
    return Path(variables.TRAINED_MODELS_PATH) / f"h{horizon:02d}_{model_name}.joblib"


def file_signature(paths):
    """Cheap artifact-lineage signature checked again by the evaluation notebook."""
    signature = []
    for path in sorted(map(Path, paths), key=lambda item: item.name):
        if not path.exists():
            raise FileNotFoundError(path)
        stat = path.stat()
        signature.append((path.name, int(stat.st_size), int(stat.st_mtime_ns)))
    return signature


all_model_paths = [
    trained_model_path(horizon, model_name)
    for horizon in HORIZON_LIST
    for _, _, model_name, _ in COMPONENT_SPECS
]
selection_paths = [
    SELECTED_FEATURES_DIR / f"FEATURES_OPTIMAL_AMOUNT_{objective}.parquet"
    for objective in FEATURE_OBJECTIVES
]

# Fail now rather than silently building a partial horizon list.
model_signature_before = file_signature(all_model_paths)
selection_signature_before = file_signature(selection_paths)
print(f"Verified {len(all_model_paths)} trained component model files")


In [ ]:
def convert_from_asinh(values):
    return (np.sinh(values) * variables.PRICE_TRANSFORM_SCALE).astype(np.float32)


def predict_components(horizon, feature_frame):
    """Load, predict, and release the eight trained models for one horizon."""
    predictions = {}
    for objective in FEATURE_OBJECTIVES:
        columns = selected_features(objective, horizon)
        missing = sorted(set(columns).difference(feature_frame.columns))
        if missing:
            raise KeyError(f"Missing features for {objective} h{horizon}: {missing[:5]}")

        X_horizon = target_features.append_target_time_feats(
            feature_frame[columns].astype(np.float32), horizon
        )
        for key, spec_objective, model_name, output_kind in COMPONENT_SPECS:
            if spec_objective != objective:
                continue
            model = joblib.load(trained_model_path(horizon, model_name))
            if hasattr(model, "set_params"):
                model.set_params(
                    n_jobs=THREADS_PER_MODEL,
                    num_threads=THREADS_PER_MODEL,
                )
            if int(model.n_features_in_) != X_horizon.shape[1]:
                raise ValueError(
                    f"{model_name} h{horizon} expects {model.n_features_in_} features; "
                    f"received {X_horizon.shape[1]}"
                )
            if output_kind == "probability":
                values = model.predict_proba(X_horizon)[:, 1].astype(np.float32)
            else:
                values = convert_from_asinh(model.predict(X_horizon))
            if not np.isfinite(values).all():
                raise ValueError(f"Non-finite predictions from {model_name} h{horizon}")
            predictions[key] = values
            del model
        del X_horizon
        gc.collect()
    return predictions


def build_meta_features(component_predictions):
    """Create a fixed, ordered stacker matrix and return its L1 anchor."""
    base_l1 = component_predictions["base_l1"]
    base_l2 = component_predictions["base_l2"]
    spike_mae = component_predictions["spike_mae"]
    spike_q90 = component_predictions["spike_q90"]
    dip_mae = component_predictions["dip_mae"]
    dip_q10 = component_predictions["dip_q10"]

    epsilon = np.float32(1e-6)
    spike_probability = np.clip(
        component_predictions["spike_probability"], epsilon, 1.0 - epsilon
    )
    dip_probability = np.clip(
        component_predictions["dip_probability"], epsilon, 1.0 - epsilon
    )
    spike_logit = np.log(spike_probability / (1.0 - spike_probability))
    dip_logit = np.log(dip_probability / (1.0 - dip_probability))

    spike_mae_gap = np.maximum(spike_mae - base_l1, 0.0)
    spike_q90_gap = np.maximum(spike_q90 - base_l1, 0.0)
    dip_mae_gap = np.minimum(dip_mae - base_l1, 0.0)
    dip_q10_gap = np.minimum(dip_q10 - base_l1, 0.0)

    meta = np.column_stack([
        base_l1,
        base_l2,
        spike_mae,
        spike_q90,
        dip_mae,
        dip_q10,
        spike_probability,
        dip_probability,
        spike_logit,
        dip_logit,
        base_l2 - base_l1,
        spike_mae_gap,
        spike_q90_gap,
        dip_mae_gap,
        dip_q10_gap,
        spike_probability * spike_mae_gap,
        spike_probability * spike_q90_gap,
        dip_probability * dip_mae_gap,
        dip_probability * dip_q10_gap,
    ]).astype(np.float32)

    if meta.shape[1] != len(META_FEATURE_NAMES):
        raise RuntimeError("Meta-feature layout does not match META_FEATURE_NAMES")
    if not np.isfinite(meta).all():
        raise ValueError("Meta-feature matrix contains non-finite values")
    return meta, base_l1


In [ ]:
def fit_postprocessor(horizon, meta, anchor, actual, component_predictions):
    return fit_horizon_postprocessor(
        horizon=horizon,
        meta=meta,
        anchor=anchor,
        actual=actual,
        index=targets_validate.index,
        component_predictions=component_predictions,
        test_start=variables.TEST_START,
        horizon_granularity_minutes=variables.HORIZON_GRANULARITY_IN_MINUTES,
        feature_granularity_minutes=variables.FEATURE_GRANULARITY_IN_MINUTES,
        stacker_params=STACKER_PARAMS,
        spike_threshold=variables.SPIKE_THRESHOLD,
        **POSTPROCESSOR_PARAMS,
    )


## Fit one central stacker and spike router per horizon

The artifact records the exact trained-model and selected-feature file
signatures. Notebook 4 refuses to evaluate it if those upstream artifacts have
changed, preventing stale model/combiner combinations.


In [ ]:
stackers = []
spike_routers = []
validation_diagnostics = []

for horizon in tqdm(HORIZON_LIST, desc="Fitting post-training models", unit="horizon"):
    component_predictions = predict_components(horizon, features_validate)
    meta, anchor = build_meta_features(component_predictions)
    actual = targets_validate[f"target_h{horizon}"].to_numpy(dtype=np.float32)
    stacker, spike_router, diagnostics = fit_postprocessor(
        horizon,
        meta,
        anchor,
        actual,
        component_predictions,
    )
    stackers.append(stacker)
    spike_routers.append(spike_router)
    validation_diagnostics.append(diagnostics)
    del component_predictions, meta, anchor, actual, stacker, spike_router
    gc.collect()

# Ensure no model or feature-selection artifact changed during this long run.
if file_signature(all_model_paths) != model_signature_before:
    raise RuntimeError("A trained model changed while stackers were being built")
if file_signature(selection_paths) != selection_signature_before:
    raise RuntimeError("Selected-feature metadata changed while stackers were being built")

artifact = {
    "schema_version": 3,
    "approach": "purged_residual_stacking_with_calibrated_spike_router",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "target_region": variables.TARGET_REGION,
    "horizon_list": HORIZON_LIST,
    "component_specs": COMPONENT_SPECS,
    "meta_feature_names": META_FEATURE_NAMES,
    "stacker_params": STACKER_PARAMS,
    "postprocessor_params": POSTPROCESSOR_PARAMS,
    "trained_model_signature": model_signature_before,
    "selected_feature_signature": selection_signature_before,
    "stackers": stackers,
    "spike_routers": spike_routers,
    "validation_diagnostics": validation_diagnostics,
}

variables.FINAL_PARAMS_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary_path = variables.FINAL_PARAMS_PATH.with_name(
    variables.FINAL_PARAMS_PATH.name + ".building"
)
joblib.dump(artifact, temporary_path)
os.replace(temporary_path, variables.FINAL_PARAMS_PATH)

diagnostics_df = pd.DataFrame(validation_diagnostics)
diagnostics_df.to_csv(STACKER_DIAGNOSTICS_PATH, index=False)

print(f"Saved {len(stackers)} stackers and spike routers to {variables.FINAL_PARAMS_PATH}")
print(f"Saved validation diagnostics to {STACKER_DIAGNOSTICS_PATH}")
display(diagnostics_df.head(10))
print(
    "Median audit MAE skill vs base: "
    f"{diagnostics_df['central_mae_skill_vs_base_pct'].median():.2f}%"
)
print(
    "Median audit spike-recall gain from routing: "
    f"{100 * (diagnostics_df['tail_audit_spike_recall'] - diagnostics_df['central_audit_spike_recall']).median():.2f} percentage points"
)


In [ ]:
# Return this kernel's memory to the OS before running notebook 4.
release_memory()
